In [47]:
import pyspark
import os

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import when
from pyspark import Row


In [ ]:
# Your path to postgresql driver. This path gived as an example
JDBC_DRIVER_PATH = "C:\\Apache\\driver-postgresql-42.5.6\\postgresql-42.5.6.jar"

spark = SparkSession\
        .builder\
        .appName("Spark-Task-1")\
        .master("local[*]")\
        .config("spark.jars", JDBC_DRIVER_PATH)\
        .getOrCreate()

print(spark.version)

3.2.2


In [ ]:
JDBC_URL = "jdbc:postgresql://<your_host>:<your_port>/<your_database_name>"
JDBC_DRIVER_CLASS = "org.postgresql.Driver"
DB_USER = "your_database_user"
DB_PASSWORD = "your_password"

actor = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.actor") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

address = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.address") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

film = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.film") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

film_actor = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.film_actor") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

film_category = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.film_category") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

category = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.category") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

customer = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.customer") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

inventory = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.inventory") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

rental = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.rental") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

payment = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.payment") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()

city = spark.read.format("jdbc") \
    .option("url", JDBC_URL) \
    .option("driver", JDBC_DRIVER_CLASS) \
    .option("dbtable", "public.city") \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .load()


Task 1: Output the number of movies in each category, sorted in descending order.

In [46]:
df_filtered = (
    category\
        .join(film_category, on="category_id")\
        .join(film, on="film_id")\
        .groupBy("name")\
        .agg(
            F.count("category_id").alias("films_amount")
        )
        .sort(F.desc(col("films_amount")), col("name"))
)

df_filtered.show()

+-----------+------------+
|       name|films_amount|
+-----------+------------+
|     Sports|          74|
|    Foreign|          73|
|     Family|          69|
|Documentary|          68|
|  Animation|          66|
|     Action|          64|
|        New|          63|
|      Drama|          62|
|      Games|          61|
|     Sci-Fi|          61|
|   Children|          60|
|     Comedy|          58|
|   Classics|          57|
|     Travel|          57|
|     Horror|          56|
|      Music|          51|
+-----------+------------+



Task 2: Output the 10 actors whose movies rented the most, sorted in descending order. 

In [45]:
df_joined = actor\
            .join(film_actor, on="actor_id")\
            .join(inventory, on="film_id")\
            .join(rental, on="inventory_id")

df_result = df_joined\
            .groupBy("actor_id", "first_name", "last_name")\
            .agg(
                F.count("rental_id").alias("rents_amount")
            )\
            .select("actor_id", "first_name", "last_name", "rents_amount")\
            .sort(F.col("rents_amount").desc())

df_result.show(10)

+--------+----------+-----------+------------+
|actor_id|first_name|  last_name|rents_amount|
+--------+----------+-----------+------------+
|     107|      GINA|  DEGENERES|         753|
|     181|   MATTHEW|     CARREY|         678|
|     198|      MARY|     KEITEL|         674|
|     144|    ANGELA|WITHERSPOON|         654|
|     102|    WALTER|       TORN|         640|
|      60|     HENRY|      BERRY|         612|
|     150|     JAYNE|      NOLTE|         611|
|      37|       VAL|     BOLGER|         605|
|      23|    SANDRA|     KILMER|         604|
|      90|      SEAN|    GUINESS|         599|
+--------+----------+-----------+------------+
only showing top 10 rows



Task 3: Output the category of movies on which the most money was spent. 

In [44]:
window_spec = Window.orderBy(F.col("total_cost").desc())

df_joined = category\
            .join(film_category, on="category_id")\
            .join(inventory, on="film_id")\
            .join(rental, on="inventory_id")\
            .join(payment, on="rental_id")

df_result = df_joined\
            .groupBy("name")\
            .agg(
                F.sum("amount").alias("total_cost"),
            )\
            .withColumn(
                    "rnk",
                    F.rank().over(window_spec)
            )\
            .sort(F.col("rnk").asc())\

df_result.show(1, truncate=False)

+------+----------+---+
|name  |total_cost|rnk|
+------+----------+---+
|Sports|5214.43   |1  |
+------+----------+---+
only showing top 1 row



Task 4: Output the names of movies that are not in the inventory. 

In [ ]:
df_result = film\
            .join(inventory, on="film_id", how="left_anti")\
            .select("film_id", "title")\
            .sort(col("title"))

df_result.show(truncate=False)

+-------+----------------------+
|film_id|title                 |
+-------+----------------------+
|14     |ALICE FANTASIA        |
|33     |APOLLO TEEN           |
|36     |ARGONAUTS TOWN        |
|38     |ARK RIDGEMONT         |
|41     |ARSENIC INDEPENDENCE  |
|87     |BOONDOCK BALLROOM     |
|108    |BUTCH PANTHER         |
|128    |CATCH AMISTAD         |
|144    |CHINATOWN GLADIATOR   |
|148    |CHOCOLATE DUCK        |
|171    |COMMANDMENTS EXPRESS  |
|192    |CROSSING DIVORCE      |
|195    |CROWDS TELEMARK       |
|198    |CRYSTAL BREAKING      |
|217    |DAZED PUNK            |
|221    |DELIVERANCE MULHOLLAND|
|318    |FIREHOUSE VIETNAM     |
|325    |FLOATS GARDEN         |
|332    |FRANKENSTEIN STRANGER |
|359    |GLADIATOR WESTWARD    |
+-------+----------------------+
only showing top 20 rows



Task 5: Output the top 3 actors who have appeared most in movies in the “Children” category. If several actors have the same number of movies, output all of them. 

In [39]:
window_spec = Window.orderBy(F.desc("amount"))

df_joined_and_aggregated = actor\
            .join(film_actor, on="actor_id")\
            .join(film_category, on="film_id")\
            .join(category, on="category_id")\
            .filter(col("name") == "Children")\
            .groupBy("actor_id", "first_name", "last_name")\
            .agg(
                F.count("film_id").alias("amount")
            )

df_result = df_joined_and_aggregated\
            .withColumn(
                    "rnk",
                    F.dense_rank().over(window_spec)
            )\
            .filter(F.col("rnk") <= 3)\
            .sort(F.col("rnk").asc(), F.col("amount").desc())\
            .select("rnk", "actor_id", "first_name", "last_name", "amount")

df_result.show()

+---+--------+----------+---------+------+
|rnk|actor_id|first_name|last_name|amount|
+---+--------+----------+---------+------+
|  1|      17|     HELEN|   VOIGHT|     7|
|  2|     140|    WHOOPI|     HURT|     5|
|  2|      80|     RALPH|     CRUZ|     5|
|  2|     127|     KEVIN|  GARLAND|     5|
|  2|      66|      MARY|    TANDY|     5|
|  3|      81|  SCARLETT|    DAMON|     4|
|  3|      23|    SANDRA|   KILMER|     4|
|  3|     109| SYLVESTER|     DERN|     4|
|  3|     187|     RENEE|     BALL|     4|
|  3|      92|   KIRSTEN|   AKROYD|     4|
|  3|     101|     SUSAN|    DAVIS|     4|
|  3|      13|       UMA|     WOOD|     4|
|  3|      58| CHRISTIAN|   AKROYD|     4|
|  3|      93|     ELLEN|  PRESLEY|     4|
|  3|     173|      ALAN| DREYFUSS|     4|
|  3|     150|     JAYNE|    NOLTE|     4|
|  3|     131|      JANE|  JACKMAN|     4|
|  3|     142|      JADA|    RYDER|     4|
|  3|      37|       VAL|   BOLGER|     4|
+---+--------+----------+---------+------+



Task 6: Output cities with the number of active and inactive customers (active - customer.active = 1). Sort by the number of inactive customers in descending order.

In [41]:
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col

df_joined = city\
            .join(address, on="city_id")\
            .join(customer, on="address_id")

df_result = df_joined\
            .groupBy("city")\
            .agg(
                F.sum(
                    F.when(col("active") == 0, 1).otherwise(0)
            ).alias("inactive_customers"),
                F.sum(
                    F.when(col("active") == 1, 1).otherwise(0)
            ).alias("active_customers")
            )\
            .sort(col("inactive_customers").desc())

df_result.show()

+------------------+------------------+----------------+
|              city|inactive_customers|active_customers|
+------------------+------------------+----------------+
|         Pingxiang|                 1|               0|
|       Szkesfehrvr|                 1|               0|
|  Charlotte Amalie|                 1|               0|
|         Najafabad|                 1|               0|
|           Wroclaw|                 1|               0|
|            Ktahya|                 1|               0|
|           Bat Yam|                 1|               0|
|   Southend-on-Sea|                 1|               0|
|            Amroha|                 1|               0|
|            Kamyin|                 1|               0|
|          Xiangfan|                 1|               0|
|            Daxian|                 1|               0|
|          Uluberia|                 1|               0|
|     Coatzacoalcos|                 1|               0|
|        Kumbakonam|           

Task 7: Output the category of movies that have the highest number of total rental hours in the cities (customer.address_id in this city), and that start with the letter “a”. Do the same for cities with a “-” symbol.

In [ ]:
df_joined = (
    city
        .join(address, on="city_id")
        .join(customer, on="address_id")
        .join(rental, on="customer_id")
        .join(inventory, on="inventory_id")
        .join(film_category, on="film_id")
        .join(category, on="category_id")
        .select(
            F.col("name").alias("category_name"),
            F.col("city"),
            F.col("rental_date"),
            F.col("return_date")
        )
        .filter(F.col("return_date").isNotNull())
        .withColumn(
            "rental_seconds",
            F.unix_timestamp(F.col("return_date")) - F.unix_timestamp(F.col("rental_date"))
        )
)

df_agg = df_joined.groupBy("category_name").agg(
    F.sum(
        F.when(
            F.lower(F.col("city")).like("a%"),
            F.col("rental_seconds")
        )
    ).alias("total_seconds_a_city"),
    F.sum(
        F.when(
            col("city").contains("-"),
            F.col("rental_seconds")
        )
    ).alias("total_seconds_dash_city")
)

df_total_hours = df_agg.withColumn(
    "total_hours_a_city",
    F.round(F.col("total_seconds_a_city") / 3600, 2)
).withColumn(
    "total_hours_dash_city",
    F.round(F.col("total_seconds_dash_city") / 3600, 2)
).drop(
    "total_seconds_a_city",
    "total_seconds_dash_city"
).sort(F.col("category_name"))

window_spec_a = Window.orderBy(F.col("total_hours_a_city").desc())
window_spec_dash = Window.orderBy(F.col("total_hours_dash_city").desc())

df_ranked = df_total_hours\
        .withColumn(
            "rank_a",
            F.rank().over(window_spec_a)
        )\
        .withColumn(
            "rank_dash",
            F.rank().over(window_spec_dash)
        )

df_ranked\
    .filter(F.col("rank_a") == 1)\
    .select(F.col("category_name"), F.col("total_hours_a_city"))\
    .show()

df_ranked\
    .filter(F.col("rank_dash") == 1)\
    .select(F.col("category_name"), F.col("total_hours_dash_city"))\
    .show()


+-------------+------------------+
|category_name|total_hours_a_city|
+-------------+------------------+
|       Sports|          12360.35|
+-------------+------------------+

+-------------+---------------------+
|category_name|total_hours_dash_city|
+-------------+---------------------+
|      Foreign|              6472.15|
+-------------+---------------------+

